# 🔥 Fire Classification Using MODIS Data (India 2021–2023)
Final Submission Notebook

**Steps Covered:**
1. Data Cleaning
2. SMOTE for Class Imbalance
3. Model Training & Evaluation
4. Confusion Matrix & Reports
5. Save Best Model

In [10]:



#  Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import pickle
import os


In [11]:
#  Load Multiple Years of MODIS Fire Data
df1 = pd.read_csv("modis_2021_India.csv")
df2 = pd.read_csv("modis_2022_India.csv")
df3 = pd.read_csv("modis_2023_India.csv")


In [12]:
#  Combine and Clean Data
df = pd.concat([df1, df2, df3])
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)


In [13]:
# ✅ Check Class Distribution Before SMOTE
print("Class distribution before SMOTE:")
print(df['type'].value_counts())


Class distribution before SMOTE:
type
0    257625
2     13550
3        42
Name: count, dtype: int64


In [14]:
from sklearn.preprocessing import StandardScaler

# ✅ Feature Selection
X = df[['brightness', 'scan', 'track', 'acq_time', 'latitude', 'longitude']]
y = df['type']  # fixed from 'fire_type' to 'type'

# ✅ Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [15]:
#  SMOTE for Balancing
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_scaled, y)

#  After SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_resampled).value_counts())


Class distribution after SMOTE:
type
0    257625
2    257625
3    257625
Name: count, dtype: int64


In [16]:
#  Split Data
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

#  Model Training
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [17]:
# Evaluation
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9897137311984474
[[50839   612     3]
 [  974 50695     0]
 [    0     1 51451]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     51454
           2       0.99      0.98      0.98     51669
           3       1.00      1.00      1.00     51452

    accuracy                           0.99    154575
   macro avg       0.99      0.99      0.99    154575
weighted avg       0.99      0.99      0.99    154575



In [18]:
# Save Model
with open('best_fire_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print(" Model saved successfully as best_fire_model.pkl")


 Model saved successfully as best_fire_model.pkl
